## Parte 1 — Softmax con temperatura

La softmax con temperatura convierte el vector de logits en probabilidades mediante p_i = exp(z_i/T) / Σ_j exp(z_j/T). Para que el cálculo sea numéricamente estable, la celda resta el máximo de los logits antes de exponenciar: p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T). Esta resta se cancela entre numerador y denominador, de modo que el resultado no cambia, pero se evitan desbordamientos cuando los logits o las temperaturas producen exponenciales muy grandes. Con los logits t0..t5 = 2.0, 1.0, 0.5, 0.2, −1.0, −3.0, la celda calcula la distribución para T = 0.5, T = 1 y T = 2, verificando que cada una suma 1.0, y su entropía en bits, H = −Σ p_i log2(p_i).

| Token | T = 0.5 | T = 1.0 | T = 2.0 |
|---|---:|---:|---:|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | **0.9147** | **1.7998** | **2.2888** |

La temperatura actúa como control de concentración: al subir T de 0.5 a 2, la masa del token dominante t0 cae de 0.8231 a 0.3583 mientras que la de los tokens poco probables sube (t5 pasa de 0.0000 a 0.0294). En consecuencia, la entropía crece monótonamente con T: 0.9147 → 1.7998 → 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
# =====================================================================
# SUBTAREA T1: Softmax con temperatura (NumPy, estable) + entropía en bits
# Tarea B — MMIA 6013 · Taller 03 v2
# =====================================================================
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Datos del enunciado
# ---------------------------------------------------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TEMPERATURAS = [0.5, 1.0, 2.0]

# ---------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (max-subtraction):
# p_i = exp(z_i/T) / sum_j exp(z_j/T), restando el máximo de los logits
# ---------------------------------------------------------------------
def softmax_con_temperatura(logits, T):
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - np.max(z)          # estabilidad: evita overflow de exp
    e = np.exp(z)
    return e / np.sum(e)

def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]               # evita log2(0)
    return float(-np.sum(p * np.log2(p)))

# ---------------------------------------------------------------------
# Cálculo de distribuciones, entropías y verificación de suma = 1
# ---------------------------------------------------------------------
distribuciones, entropias, sumas = {}, {}, {}
for T in TEMPERATURAS:
    p = softmax_con_temperatura(LOGITS, T)
    key = f"T={T}"
    distribuciones[key] = p
    entropias[key] = entropia_bits(p)
    sumas[key] = float(np.sum(p))

# Comprobación de estabilidad: desplazar los logits +1000 no cambia nada
estabilidad_ok = bool(np.allclose(
    softmax_con_temperatura(LOGITS + 1000.0, 1.0),
    distribuciones["T=1.0"], rtol=0, atol=1e-12))

# ---------------------------------------------------------------------
# Tabla con cuatro decimales (filas: tokens y entropía; columnas: T)
# ---------------------------------------------------------------------
datos = {}
for T in TEMPERATURAS:
    key = f"T={T}"
    datos[key] = [f"{v:.4f}" for v in distribuciones[key]] + [f"{entropias[key]:.4f}"]
tabla = pd.DataFrame(datos, index=TOKENS + ["Entropía (bits)"])

print("Tabla (4 decimales):")
print(tabla.to_string())

# ---------------------------------------------------------------------
# Figura: distribuciones para las tres temperaturas
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(TOKENS))
ancho = 0.25
for i, T in enumerate(TEMPERATURAS):
    key = f"T={T}"
    ax.bar(x + (i - 1) * ancho, distribuciones[key], width=ancho,
           label=f"T={T}  (H = {entropias[key]:.4f} bits)")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad $p_i$")
ax.set_title("Softmax con temperatura — logits (2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ---------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ---------------------------------------------------------------------
tabla_json = [
    {"fila": fila, **{col: tabla.loc[fila, col] for col in tabla.columns}}
    for fila in tabla.index
]

resultados = {
    "subtarea": "T1_softmax_con_temperatura",
    "tokens": TOKENS,
    "logits": {tok: float(z) for tok, z in zip(TOKENS, LOGITS)},
    "temperaturas": TEMPERATURAS,
    "distribuciones": {k: v.tolist() for k, v in distribuciones.items()},
    "suma_distribuciones": sumas,
    "entropias_bits": entropias,
    "softmax_estable_max_subtraction": estabilidad_ok,
    "tabla_4_decimales": tabla_json,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------------
# Resumen breve
# ---------------------------------------------------------------------
print("\nResumen:")
for T in TEMPERATURAS:
    key = f"T={T}"
    print(f"  {key}: suma={sumas[key]:.6f} | H = {entropias[key]:.4f} bits")
print(f"  Estabilidad (logits +1000 sin cambio): {estabilidad_ok}")
print("Archivos generados: resultados.json, t1_softmax_temperatura.png")


Tabla (4 decimales):
                  T=0.5   T=1.0   T=2.0
t0               0.8231  0.5516  0.3583
t1               0.1114  0.2029  0.2173
t2               0.0410  0.1231  0.1693
t3               0.0225  0.0912  0.1457
t4               0.0020  0.0275  0.0800
t5               0.0000  0.0037  0.0294
Entropía (bits)  0.9147  1.7998  2.2888

Resumen:
  T=0.5: suma=1.000000 | H = 0.9147 bits
  T=1.0: suma=1.000000 | H = 1.7998 bits
  T=2.0: suma=1.000000 | H = 2.2888 bits
  Estabilidad (logits +1000 sin cambio): True
Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1, top-p con p = 0.9 ordena los tokens de mayor a menor probabilidad, acumula en ese orden y conserva el conjunto mínimo de tokens cuya probabilidad acumulada alcanza 0.9 (el primer k tal que la suma de las k mayores probabilidades es ≥ 0.9). Los tokens fuera de ese núcleo se descartan y las probabilidades de los supervivientes se renormalizan dividiéndolas entre la masa total del núcleo.

| Token | p (T = 1) | Acumulada | ¿En el núcleo? | p renormalizada |
|---|---:|---:|:---:|---:|
| t0 | 0.5516 | 0.5516 | Sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | Sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | Sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | Sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | No | — |
| t5 | 0.0037 | 1.0000 | No | — |

Con solo t0, t1 y t2 la acumulada es 0.8776 < 0.9, por lo que es necesario incluir t3: el núcleo mínimo queda formado por **t0, t1, t2 y t3**, con una masa acumulada de 0.9688. Los tokens descartados son **t4 y t5**. La distribución renormalizada es t0 = 0.5694, t1 = 0.2095, t2 = 0.1270, t3 = 0.0941 (suma 1.0). Todas las probabilidades supervivientes aumentan respecto de las originales (p. ej., t0 pasa de 0.5516 a 0.5694) porque entre ellas se reparte la masa de los tokens descartados t4 y t5.

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1.

Entrada : entrada/T1/resultados.json  (distribución softmax con T=1 de la subtarea T1)
Salidas : resultados.json (contrato de la subtarea), top_p_T2.png (figura)
"""

import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P_NUCLEO = 0.9

# ------------------------------------------------------------------
# 1) Cargar la distribución de T=1 obtenida en T1
# ------------------------------------------------------------------
fuente = "entrada/T1/resultados.json"
try:
    with open(fuente, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    tokens = list(t1["tokens"])
    p_T1 = np.asarray(t1["distribuciones"]["T=1.0"], dtype=float)
    origen = "entrada/T1/resultados.json (salida de T1)"
except (FileNotFoundError, KeyError, json.JSONDecodeError):
    # Respaldo defensivo: recalcular softmax(T=1) de forma numéricamente estable
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    e = np.exp(z)
    p_T1 = e / e.sum()
    origen = "recalculada con softmax estable (T=1); entrada/T1 no disponible"

# ------------------------------------------------------------------
# 2) Top-p: ordenar por probabilidad descendente y acumular
# ------------------------------------------------------------------
orden = np.argsort(-p_T1, kind="stable")          # índices de mayor a menor prob.
tokens_ord = [tokens[i] for i in orden]
probs_ord = p_T1[orden]
acum = np.cumsum(probs_ord)                        # probabilidad acumulada

# Conjunto MÍNIMO cuya acumulada alcanza p: primer k con acum[k] >= p
k = int(np.searchsorted(acum, P_NUCLEO, side="left"))
if k >= len(acum):                                 # salvaguarda por punto flotante
    k = len(acum) - 1

idx_nucleo = orden[: k + 1]
tokens_sup = [tokens[i] for i in idx_nucleo]       # tokens supervivientes
probs_sup = p_T1[idx_nucleo]
masa_nucleo = float(probs_sup.sum())

# ------------------------------------------------------------------
# 3) Renormalizar el núcleo
# ------------------------------------------------------------------
probs_ren = probs_sup / masa_nucleo
suma_ren = float(probs_ren.sum())
tokens_desc = [t for t in tokens if t not in tokens_sup]
renorm_map = {tok: float(p) for tok, p in zip(tokens_sup, probs_ren)}

# ------------------------------------------------------------------
# 4) Tabla legible (4 decimales) + resultados.json (contrato)
# ------------------------------------------------------------------
tabla = []
for tok, p, a in zip(tokens_ord, probs_ord, acum):
    fila = {
        "token": tok,
        "prob_T1": f"{float(p):.4f}",
        "prob_acumulada": f"{float(a):.4f}",
        "en_nucleo": tok in renorm_map,
        "prob_renormalizada": (f"{renorm_map[tok]:.4f}" if tok in renorm_map else None),
    }
    tabla.append(fila)

resultados = {
    "subtarea": "T2_top_p_nucleus_sampling_p0.9_sobre_T1",
    "p_nucleo": P_NUCLEO,
    "fuente_distribucion": origen,
    "tokens": tokens,
    "distribucion_T1_entrada": {tok: float(p) for tok, p in zip(tokens, p_T1)},
    "orden_descendente": tokens_ord,
    "probs_ordenadas": [float(x) for x in probs_ord],
    "prob_acumulada_ordenada": [float(x) for x in acum],
    "criterio_conjunto_minimo": "primer k tal que la suma de las k+1 mayores probabilidades >= 0.9",
    "tokens_supervivientes": tokens_sup,
    "num_tokens_supervivientes": len(tokens_sup),
    "masa_acumulada_nucleo": masa_nucleo,
    "tokens_descartados": tokens_desc,
    "distribucion_renormalizada": renorm_map,
    "distribucion_renormalizada_orden_tokens": tokens_sup,
    "distribucion_renormalizada_valores": [float(x) for x in probs_ren],
    "suma_distribucion_renormalizada": suma_ren,
    "tabla_4_decimales": tabla,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------------
# 5) Figura: distribución original (con acumulada y umbral 0.9) y núcleo renormalizado
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

x = np.arange(len(tokens))
axes[0].bar(x, p_T1, color="steelblue", label="p_i (T=1)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens)
axes[0].set_ylabel("Probabilidad")
axes[0].set_title("Distribución de T=1 (entrada de T1)")
axes[0].set_ylim(0, max(p_T1) * 1.20)
for i, p in enumerate(p_T1):
    axes[0].text(i, p + 0.008, f"{p:.4f}", ha="center", fontsize=8)
ax2 = axes[0].twinx()
ax2.plot(x, acum, "o--", color="crimson", label="acumulada")
ax2.axhline(P_NUCLEO, color="black", ls=":", lw=1.2)
ax2.text(len(tokens) - 0.5, P_NUCLEO + 0.015, "p = 0.9", ha="right", fontsize=9)
ax2.set_ylabel("Probabilidad acumulada")
ax2.set_ylim(0, 1.08)
axes[0].axvspan(k - 0.5, k + 0.5, color="orange", alpha=0.15)

x2 = np.arange(len(tokens_sup))
axes[1].bar(x2, probs_ren, color="darkorange")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(tokens_sup)
axes[1].set_ylabel("Probabilidad renormalizada")
axes[1].set_title(f"Núcleo top-p (p=0.9): {tokens_sup}")
axes[1].set_ylim(0, max(probs_ren) * 1.20)
for i, p in enumerate(probs_ren):
    axes[1].text(i, p + 0.008, f"{p:.4f}", ha="center", fontsize=8)

fig.suptitle("T2 — Muestreo de núcleo (top-p, p=0.9) sobre la distribución de T=1", y=1.02)
fig.tight_layout()
plt.savefig("top_p_T2.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------------
print("=== T2: Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1 ===")
print(f"Fuente de la distribución: {origen}")
print("Orden descendente : " + ", ".join(f"{t}={p:.4f}" for t, p in zip(tokens_ord, probs_ord)))
print("Acumulada         : " + ", ".join(f"{a:.4f}" for a in acum))
print(f"Tokens supervivientes (conjunto mínimo con acumulada >= {P_NUCLEO}): {tokens_sup}")
print(f"Tokens descartados: {tokens_desc}")
print(f"Masa acumulada del núcleo (antes de renormalizar): {masa_nucleo:.6f}")
print("Distribución renormalizada:")
for tok, p in zip(tokens_sup, probs_ren):
    print(f"  {tok}: {p:.6f}")
print(f"Suma de la distribución renormalizada: {suma_ren:.10f}")
print("Figura guardada: top_p_T2.png | Resultados escritos en resultados.json")


=== T2: Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1 ===
Fuente de la distribución: entrada/T1/resultados.json (salida de T1)
Orden descendente : t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037
Acumulada         : 0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000
Tokens supervivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Tokens descartados: ['t4', 't5']
Masa acumulada del núcleo (antes de renormalizar): 0.968820
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.0000000000
Figura guardada: top_p_T2.png | Resultados escritos en resultados.json


## Parte 3 — Comprobación empírica

La celda toma 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0) y compara, en un gráfico de barras, las frecuencias observadas con las probabilidades teóricas.

| Token | p teórica | Conteo observado | Frecuencia observada |
|---|---:|---:|---:|
| t0 | 0.5694 | 5682 | 0.5682 |
| t1 | 0.2095 | 2093 | 0.2093 |
| t2 | 0.1270 | 1296 | 0.1296 |
| t3 | 0.0941 | 929 | 0.0929 |

La divergencia KL de la distribución empírica respecto de la teórica, definida como D_KL(P_emp ‖ P_teo) = Σ p_emp · log2(p_emp / p_teo), es de **5.006898814879229e-05 bits**. El gráfico de barras muestra ambas series prácticamente superpuestas; la mayor discrepancia se observa en t2, con 0.1296 observado frente a 0.1270 teórico. Un KL tan cercano a cero confirma que el muestreo con la semilla indicada reproduce la distribución teórica del núcleo, con desviaciones del tamaño esperable del error de Monte Carlo para n = 10 000.

In [3]:
# T3: código aprobado por el crítico
# T3 — Comprobación empírica: muestreo de la distribución renormalizada de T2,
# comparación de frecuencias observadas vs probabilidades teóricas y divergencia KL (bits).
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def cargar_distribucion_t2():
    """Lee la distribución renormalizada de T2; si no existiera, la recomputa desde los logits."""
    try:
        with open("entrada/T2/resultados.json", "r", encoding="utf-8") as f:
            t2 = json.load(f)
        tokens = t2["distribucion_renormalizada_orden_tokens"]
        p_teo = np.array(t2["distribucion_renormalizada_valores"], dtype=float)
        fuente = "entrada/T2/resultados.json (distribución renormalizada de T2)"
        return tokens, p_teo, fuente
    except (FileNotFoundError, KeyError):
        # Fallback: softmax con T=1 + top-p (p=0.9) sobre los logits del enunciado
        tokens_full = ["t0", "t1", "t2", "t3", "t4", "t5"]
        logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
        z = logits - logits.max()
        p1 = np.exp(z) / np.exp(z).sum()  # T = 1
        orden = np.argsort(-p1)
        acum = np.cumsum(p1[orden])
        k = int(np.searchsorted(acum, 0.9) + 1)  # conjunto mínimo que alcanza 0.9
        nucleo = orden[:k]
        p_nuc = p1[nucleo] / p1[nucleo].sum()
        tokens = [tokens_full[i] for i in nucleo]
        fuente = "recomputada desde logits (softmax T=1 + top-p p=0.9)"
        return tokens, p_nuc, fuente


def main():
    tokens, p_teo, fuente = cargar_distribucion_t2()
    n = 10000

    # Muestreo: 10,000 muestras con la semilla exigida
    rng = np.random.default_rng(0)
    muestras = rng.choice(len(tokens), size=n, p=p_teo)

    counts = np.bincount(muestras, minlength=len(tokens))
    freq_obs = counts / n

    # Divergencia KL (en bits) de la empírica respecto de la teórica:
    # D_KL(P_emp || P_teo) = sum_i p_emp(i) * log2(p_emp(i) / p_teo(i))
    mask = freq_obs > 0
    kl_bits = float(np.sum(freq_obs[mask] * np.log2(freq_obs[mask] / p_teo[mask])))

    # Gráfico de barras: frecuencias observadas vs probabilidades teóricas
    x = np.arange(len(tokens))
    width = 0.38
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, freq_obs, width, color="#4C72B0",
           label="Frecuencia observada (10,000 muestras)")
    ax.bar(x + width / 2, p_teo, width, color="#DD8452",
           label="Probabilidad teórica (top-p renormalizada)")
    ax.set_xticks(x)
    ax.set_xticklabels(tokens)
    ax.set_ylabel("Probabilidad / Frecuencia relativa")
    ax.set_xlabel("Token")
    ax.set_title("T3: Frecuencias observadas vs probabilidades teóricas\n"
                 "top-p (p=0.9) sobre T=1 · 10,000 muestras · default_rng(0)")
    for xi, fo, pt in zip(x, freq_obs, p_teo):
        ax.text(xi - width / 2, fo + 0.008, f"{fo:.4f}", ha="center", fontsize=8)
        ax.text(xi + width / 2, pt + 0.008, f"{pt:.4f}", ha="center", fontsize=8)
    ax.legend()
    ax.set_ylim(0, max(freq_obs.max(), p_teo.max()) * 1.15)
    plt.tight_layout()
    plt.savefig("T3_frecuencias_vs_teoricas.png", dpi=120)
    plt.close(fig)

    # Contrato de resultados
    resultados = {
        "subtarea": "T3_comprobacion_empirica_muestreo_top_p",
        "fuente_distribucion": fuente,
        "semilla": "numpy.random.default_rng(0)",
        "num_muestras": n,
        "tokens": tokens,
        "probabilidades_teoricas": p_teo.tolist(),
        "conteos_observados": counts.tolist(),
        "frecuencias_observadas": freq_obs.tolist(),
        "divergencia_KL_bits": kl_bits,
        "definicion_KL": "D_KL(P_emp || P_teo) = sum p_emp * log2(p_emp / p_teo), en bits",
        "figura": "T3_frecuencias_vs_teoricas.png",
    }
    with open("resultados.json", "w", encoding="utf-8") as f:
        json.dump(resultados, f, indent=2, ensure_ascii=False)

    # Resumen
    print("T3 — Comprobación empírica (top-p p=0.9 sobre T=1)")
    print(f"Fuente de la distribución: {fuente}")
    print(f"Muestras: {n} · Semilla: numpy.random.default_rng(0)")
    print(f"{'token':<6}{'observado':>12}{'teórico':>12}{'conteo':>10}")
    for tok, fo, pt, c in zip(tokens, freq_obs, p_teo, counts):
        print(f"{tok:<6}{fo:>12.4f}{pt:>12.4f}{c:>10d}")
    print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
    print("Figura guardada: T3_frecuencias_vs_teoricas.png")
    print("Resultados guardados: resultados.json")


if __name__ == "__main__":
    main()


T3 — Comprobación empírica (top-p p=0.9 sobre T=1)
Fuente de la distribución: entrada/T2/resultados.json (distribución renormalizada de T2)
Muestras: 10000 · Semilla: numpy.random.default_rng(0)
token    observado     teórico    conteo
t0          0.5682      0.5694      5682
t1          0.2093      0.2095      2093
t2          0.1296      0.1270      1296
t3          0.0929      0.0941       929
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: T3_frecuencias_vs_teoricas.png
Resultados guardados: resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0:** el cociente z_i/T amplifica las diferencias entre logits, de modo que el término exp(z_max/T) domina exponencialmente al resto y toda la masa se concentra en el token de mayor logit (t0, con z = 2.0). La distribución degenera en una masa de punto y su entropía tiende a 0 bits. Los datos de la Parte 1 muestran esa tendencia: al bajar T de 1 a 0.5, p(t0) sube de 0.5516 a 0.8231 y la entropía cae de 1.7998 a 0.9147 bits, marcando la dirección del límite H → 0.

**Cuando T → ∞:** ocurre lo contrario, porque z_i/T → 0 para todo i, todos los exp(z_i/T) tienden a 1 y la distribución converge a la uniforme, con p_i → 1/6 en cada uno de los seis tokens; la entropía tiende a su máximo posible, log2(6) bits. La evidencia apunta en esa dirección: a T = 2 la distribución se aplana (t0 baja a 0.3583 y t5 sube a 0.0294) y la entropía sube a 2.2888 bits, acercándose a ese máximo.

**Por qué T → 0 equivale a la decodificación voraz:** la decodificación greedy selecciona argmax_i z_i, que coincide con el argmax de la softmax por ser esta una función monótona de los logits. En el límite T → 0 la softmax asigna probabilidad 1 precisamente a ese token, de modo que muestrear de esa distribución devuelve siempre el argmax: una elección determinista idéntica a la voraz. La entropía es la evidencia cuantitativa de esta equivalencia: H → 0 bits significa incertidumbre nula, es decir, una distribución de la que solo puede salir un resultado, exactamente el token que elegiría greedy. La caída medida de la entropía al reducir la temperatura (1.7998 bits en T = 1 frente a 0.9147 bits en T = 0.5) ilustra esa concentración progresiva hacia el régimen determinista.